# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [2]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5008 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [24]:
import functools

def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        for arg in list(args) + list(kwargs.values()):
            if isinstance(arg, list):
                print(f"Długość listy: {len(arg)}")
        
        return result
    return wrapper

# Test:
@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")

process_data([1, 2, 3, 4], "Dane z args")
process_data(data_list=[5, 6, 7], name="Dane z kwargs")

Przetwarzanie Dane z args
Długość listy: 4
Przetwarzanie Dane z kwargs
Długość listy: 3


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [9]:
import functools
from datetime import datetime

def logger(filename):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start_time = datetime.now()
            result = func(*args, **kwargs)
            end_time = datetime.now()
            execution_time = (end_time - start_time).total_seconds()
            current_date = start_time.strftime("%Y-%m-%d %H:%M:%S")
            with open(filename, 'a') as f:
                f.write(f"Funkcja: {func.__name__}, Data: {current_date}, Czas wykonania: {execution_time:.4f}s\n")
            return result
        return wrapper
    return decorator



@logger("test.log")
def example_function():
    time.sleep(0.1)
    return

example_function()

NameError: name 'time' is not defined

--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [6]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [17]:
import logging


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class AccessLogger:
    def __init__(self):
        self._value = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        logging.info(f"Odczyt atrybutu {self.name}: {self._value}")
        return self._value

    def __set__(self, instance, value):
        logging.info(f"Zapis atrybutu {self.name}: {value}")
        self._value = value

class Uzytkownik:
    imie = AccessLogger()
    wiek = AccessLogger()
    
    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek


u = Uzytkownik("Piotr", 25)
print(u.imie)
u.wiek = 26
print(u.wiek)

2026-04-10 22:58:55,251 - INFO - Zapis atrybutu imie: Piotr
2026-04-10 22:58:55,252 - INFO - Zapis atrybutu wiek: 25
2026-04-10 22:58:55,252 - INFO - Odczyt atrybutu imie: Piotr
2026-04-10 22:58:55,253 - INFO - Zapis atrybutu wiek: 26
2026-04-10 22:58:55,254 - INFO - Odczyt atrybutu wiek: 26


Piotr
26


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [25]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [ ]:
def collatz_generator(n):
    while n != 1:
        yield n
        if n % 2 == 0:
            n = n // 2
        else:
            n = 3 * n + 1
    yield 1


coll = collatz_generator(10)
print(list(coll))

[10, 5, 16, 8, 4, 2, 1]


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [ ]:
import functools

current_user = {"username": "admin", "role": "superuser"}

def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user.get("role") != role:
                raise PermissionError(f"Brak wymaganej roli: {role}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@require_role("superuser")
def superuser_action():
    return "Akcja wykonana przez superusera"


try:
    print(superuser_action()) 
except PermissionError as e:
    print(e)

current_user["role"] = "user"

try:
    print(superuser_action()) 
except PermissionError as e:
    print(e)


Akcja wykonana przez superusera
Brak wymaganej roli: superuser


### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [ ]:
class Typed:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(f"Atrybut {self.name} musi być typu {self.expected_type.__name__}")
        instance.__dict__[self.name] = value

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)


class Person:
    name = Typed(str)
    age = Typed(int)

p = Person()

p.name = "Jan"
# p.name = 123

p.age = 25
# p.age = "25"

TypeError: Atrybut age musi być typu int

### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [20]:
def is_prime(n):
    # Numbers less than 2 are not prime
    if n < 2:
        return False
    # 2 is the smallest prime number
    if n == 2:
        return True
    # Follow the alternate definition
    for i in range (2, n):
        if (n%i) == 0:
            return False # n is not prime
    return True # n is prime

def prime_generator():
    n = 2
    while True:
        if is_prime(n):
            yield n
        n += 1


primes_ending_with_7 = (p for p in prime_generator() if str(p).endswith('7'))

for _ in range(5):
    print(next(primes_ending_with_7))

7
17
37
47
67
